# 6. Production Pipeline: From One Document to Many

In Section 4 we built a complete RAG pipeline — ingestion, chunking, embedding, retrieval, generation — against a single document. In Section 5 we evaluated the results.

Now the question every customer asks: **"What happens when we point this at the real corpus?"**

The answer: the pipeline doesn't change. The input does.

What *does* change at scale:
- **Source attribution** — which document did this answer come from?
- **Progress visibility** — processing 50 PDFs takes time; you need to see where you are
- **Collection management** — naming, versioning, re-indexing
- **Data conflicts** — multiple documents may say different things about the same topic

This section walks through each of these.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base

import requests
import json
import numpy as np
import chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
from docling.document_converter import DocumentConverter

url_chat = f"{endpoint_base}/chat/completions"
model_id = "granite-3-2-8b-instruct"

print("\u2705 All imports loaded")

✅ All imports loaded


## Helper Functions

These are the same `chunk_text` and `add_to_collection` functions from Section 4 — unchanged. The pipeline is the same; we're just feeding it more documents.

In [2]:
def chunk_text(text, chunk_size=1000, overlap=200):
    """Split text into chunks with overlap, respecting word boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        # Don't cut mid-word: back up to the last space
        if end < len(text):
            space_pos = text.rfind(" ", start, end)
            if space_pos > start:
                end = space_pos
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        # Ensure we always advance by at least 1 character
        next_start = end - overlap
        start = max(next_start, start + 1)
    return chunks

In [3]:
def add_to_collection(collection, chunks, embeddings, metadatas=None, ids=None, batch_size=5000):
    """Batch-load chunks into a ChromaDB collection."""
    for i in range(0, len(chunks), batch_size):
        end = min(i + batch_size, len(chunks))
        batch_kwargs = {
            "documents": chunks[i:end],
            "embeddings": embeddings[i:end].tolist() if hasattr(embeddings, 'tolist') else embeddings[i:end],
            "ids": ids[i:end] if ids else [f"chunk_{j}" for j in range(i, end)],
        }
        if metadatas:
            batch_kwargs["metadatas"] = metadatas[i:end]
        collection.add(**batch_kwargs)
    print(f"\u2705 Loaded {collection.count()} chunks into the vector store")

## 6.1 Multi-Document Pipeline

The key architectural decisions for multi-document ingestion:

1. **Unique chunk IDs** — each chunk needs an ID that encodes its source document, so we can trace any retrieval result back to its origin.
2. **Metadata tracking** — every chunk carries `{"source": filename}` so retrieval results include provenance.
3. **New collection** — we use `rpg_rules_multi` to keep this separate from the single-document collection in Section 4.

> **Facilitator note:** If you've placed additional PDFs in `docs/`, they'll be picked up automatically. The pipeline is input-agnostic.

In [ ]:
docs_dir = Path("../docs")
pdf_files = sorted(docs_dir.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF(s) in {docs_dir.resolve()}:\n")
for pdf in pdf_files:
    size_mb = pdf.stat().st_size / (1024 * 1024)
    print(f"  {pdf.name:<50} {size_mb:>8.1f} MB")

if not pdf_files:
    print("\u26a0\ufe0f  No PDFs found. Place PDF files in the docs/ directory and re-run.")

### Load the Embedding Model

We load the embedding model once and reuse it for all documents. Same Granite Embedding model from Section 4.

In [ ]:
print("Loading Granite embedding model...")
embed_model = SentenceTransformer("../models/granite-embedding-30m-english")
print("\u2705 Embedding model loaded")

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(do_ocr=False)
converter = DocumentConverter(
    format_options={InputFormat.PDF: {"pipeline_options": pipeline_options}}
)
print("\u2705 Docling converter configured (OCR disabled)")

In [ ]:
all_chunks = []
all_embeddings = []
all_metadatas = []
all_ids = []

for pdf_idx, pdf_path in enumerate(pdf_files, 1):
    filename = pdf_path.name
    print(f"\n{'='*60}")
    print(f"[{pdf_idx}/{len(pdf_files)}] Processing: {filename}")
    print(f"{'='*60}")

    # Step 1: Convert PDF to markdown with Docling
    print("  Converting PDF to markdown...")
    result = converter.convert(str(pdf_path))
    markdown_text = result.document.export_to_markdown()
    print(f"  Extracted {len(markdown_text):,} characters")

    # Step 2: Chunk the text
    chunks = chunk_text(markdown_text, chunk_size=1000, overlap=200)
    print(f"  Created {len(chunks)} chunks")

    # Step 3: Embed all chunks
    print("  Embedding chunks...")
    embeddings = embed_model.encode(chunks, show_progress_bar=False, batch_size=64)

    # Step 4: Build metadata and IDs with source attribution
    metadatas = [{"source": filename} for _ in chunks]
    ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]

    all_chunks.extend(chunks)
    all_embeddings.append(embeddings)
    all_metadatas.extend(metadatas)
    all_ids.extend(ids)

    print(f"  \u2705 Done \u2014 running total: {len(all_chunks)} chunks from {pdf_idx} document(s)")

# Stack all embeddings into a single array
all_embeddings = np.vstack(all_embeddings)

print(f"\n{'='*60}")
print(f"Pipeline complete: {len(all_chunks)} total chunks from {len(pdf_files)} document(s)")
print(f"Embedding matrix shape: {all_embeddings.shape}")
print(f"{'='*60}")

### 6.1.5 Load into ChromaDB

We create a new collection called `rpg_rules_multi`. This keeps it separate from the single-document `rpg_rules` collection built in Section 4, so we can compare them later.

Every chunk now carries metadata with its source filename — this is what enables source attribution at query time.

In [ ]:
client = chromadb.PersistentClient(path="../chroma_db")

# Delete if exists (safe for re-runs)
try:
    client.delete_collection("rpg_rules_multi")
except ValueError:
    pass

collection_multi = client.create_collection(
    name="rpg_rules_multi",
    metadata={"hnsw:space": "cosine"}
)

add_to_collection(
    collection_multi,
    all_chunks,
    all_embeddings,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"Collection '{collection_multi.name}' — {collection_multi.count()} chunks")

## 6.2 Retrieval with Source Attribution

In a single-document pipeline, you know where every answer comes from — there's only one source. With multiple documents, the customer will always ask:

> *"Where did that come from?"*

Source attribution isn't a nice-to-have. It's the difference between a demo and a deployable system. The function below extends our RAG query to return the source document for each retrieved chunk alongside the answer.

In [ ]:
def ask_with_rag_sources(question, collection, embed_model, n_results=3):
    """RAG query that returns the answer plus source attribution for each chunk."""
    # Retrieve with metadata
    query_embedding = embed_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas"]
    )

    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    sources = [(chunk, meta.get("source", "unknown")) for chunk, meta in zip(chunks, metadatas)]

    # Build grounded prompt
    context = "\n\n---\n\n".join(chunks)
    system_msg = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so."
    )
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }

    response = requests.post(url_chat, headers=headers, json=data)
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]

    return answer, sources

In [ ]:
test_question = "What happens if a Thief fails an Open Locks attempt?"
answer, sources = ask_with_rag_sources(test_question, collection_multi, embed_model)

print(f"Question: {test_question}\n")
print(f"Answer:\n{answer}\n")
print("=" * 60)
print("SOURCE ATTRIBUTION")
print("=" * 60)
for i, (chunk_text_preview, source) in enumerate(sources, 1):
    print(f"\n--- Chunk {i} | Source: {source} ---")
    print(chunk_text_preview[:200] + "...")

## 6.3 Evaluation: Same Questions, Multi-Document Corpus

We re-run the same 5 evaluation questions from Section 5. This tells us whether the multi-document pipeline produces the same quality answers — and whether source attribution is consistent.

In [ ]:
eval_questions = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "reference_answer": "The Thief must wait until they have gained another level of experience before trying again. It may only be tried once per lock."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "reference_answer": "Elves use a d6 for hit dice because they are a combination Fighter/Magic-User class, and their hit die reflects the Magic-User side."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "reference_answer": "No. Only spell casters can use magic-user scrolls."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "reference_answer": "Each side rolls 1d6 at the beginning of each round. The side with the highest roll acts first."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "reference_answer": "The number of retainers is based on the character's Charisma bonus. A character with average Charisma can hire up to 4."
    },
]

print(f"Evaluation set: {len(eval_questions)} questions")
for q in eval_questions:
    print(f"  [{q['id']}] {q['question']}")

In [ ]:
eval_results = []

print("Running evaluation against multi-document collection...\n")
for q in eval_questions:
    print(f"[{q['id']}] {q['question']}")
    answer, sources = ask_with_rag_sources(q["question"], collection_multi, embed_model)
    eval_results.append({
        "id": q["id"],
        "question": q["question"],
        "reference_answer": q["reference_answer"],
        "answer": answer,
        "sources": sources
    })
    print(f"  \u2705 Done\n")

print(f"All {len(eval_results)} questions complete.")

In [ ]:
print("=" * 80)
print("MULTI-DOCUMENT EVALUATION RESULTS")
print("=" * 80)

for r in eval_results:
    print(f"\n{'\u2500' * 80}")
    print(f"{r['id'].upper()}: {r['question']}")
    print(f"{'\u2500' * 80}")

    print(f"\n\U0001f4d6 Reference:\n   {r['reference_answer']}")
    print(f"\n\U0001f4da RAG Answer (multi-doc):\n   {r['answer']}")

    print(f"\n   Sources:")
    source_files = set(src for _, src in r["sources"])
    for src in source_files:
        print(f"     - {src}")

    print(f"\n   Retrieved chunks:")
    for i, (chunk_preview, source) in enumerate(r["sources"], 1):
        print(f"     [{i}] ({source}) {chunk_preview[:120]}...")

## 6.4 Comparison: Single-Document vs Multi-Document

Now we compare the `rpg_rules` collection from Section 4 (single document) against `rpg_rules_multi` (full corpus). This reveals what changes when you scale:

- **Redundancy** — multiple documents may contain overlapping information, leading to duplicate or near-duplicate chunks in retrieval
- **Noise** — more documents means more chunks competing for the top-N retrieval slots; irrelevant chunks from unrelated documents can displace relevant ones
- **Conflict** — different documents may state different rules, versions, or interpretations of the same concept

In [ ]:
# Load the single-doc collection from Section 4
try:
    collection_single = client.get_collection(name="rpg_rules")
    print(f"Single-doc collection: {collection_single.count()} chunks")
    print(f"Multi-doc collection:  {collection_multi.count()} chunks")
except Exception as e:
    print(f"\u26a0\ufe0f  Could not load single-doc collection 'rpg_rules': {e}")
    print("   Run Section 4 first to create it.")
    collection_single = None

if collection_single:
    compare_question = "How is initiative determined in combat?"

    print(f"\n{'=' * 70}")
    print(f"COMPARISON: {compare_question}")
    print(f"{'=' * 70}")

    # Single-doc answer (no source metadata)
    q_emb = embed_model.encode(compare_question).tolist()
    single_results = collection_single.query(
        query_embeddings=[q_emb], n_results=3
    )
    single_context = "\n\n---\n\n".join(single_results["documents"][0])
    headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    single_data = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": "Answer the question using only the provided context. If the context does not contain enough information, say so."},
            {"role": "user", "content": f"Context:\n{single_context}\n\nQuestion: {compare_question}"}
        ],
        "max_tokens": 300, "temperature": 0
    }
    resp = requests.post(url_chat, headers=headers, json=single_data)
    single_answer = resp.json()["choices"][0]["message"]["content"]

    # Multi-doc answer (with source attribution)
    multi_answer, multi_sources = ask_with_rag_sources(
        compare_question, collection_multi, embed_model
    )

    print(f"\n\U0001f4c4 Single-document RAG ({collection_single.count()} chunks):")
    print(f"   {single_answer}")

    print(f"\n\U0001f4da Multi-document RAG ({collection_multi.count()} chunks):")
    print(f"   {multi_answer}")

    print(f"\n   Sources used:")
    for _, source in multi_sources:
        print(f"     - {source}")

## 6.5 Takeaways

1. **The pipeline is input-agnostic.** The same chunking, embedding, and retrieval code processes one document or a hundred. What changes is the input list, not the architecture.

2. **Source attribution isn't optional.** The moment you move beyond a single document, every answer needs to say where it came from. Without this, the system is a black box — and no customer will deploy a black box.

3. **Scaling reveals data management problems, not model problems.** Conflicting information across documents, version drift, duplicate content — these are corpus curation issues. The model can only work with what retrieval gives it.

4. **The evaluation framework is now essential.** With a single document, you could eyeball quality. With a corpus, you need structured evaluation (Section 5) to catch regressions. If adding a new document makes existing answers worse, you need to know immediately.

> *"The pipeline scales. The question is whether your data does."*